# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syed-Attique/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

**Search question:** *Which pages should a reviewer look at first this month, because their search visibility is falling and enough demand is at stake to be worth the review time?*

- **Unit of analysis:** one page (one `content_id`) at one decision date, described by its recent search and engagement signals.
- **Output:** a ranked review queue. Each row has a score, a suggested review type (refresh, check CTR, monitor, ...) and reason codes a human can read.

**Why this lane over the others.** The starter data shows that "declining" is the majority state, not a rare one, so a yes/no flag cannot tell a reviewer where to start; only a ranking can. Lane 4 (CTR) looked weaker: a fixed CTR threshold flags most eligible pages too. Lane 1 (signals) is a good fit as a *step inside* this lane, not as the whole project. The AI-referral direction is ruled out for a classifier because only a small share of pages have any AI sessions. I can confirm or change this choice until the end of Week 4.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Syed-Attique/FlyRank-Internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    # Colab starts with an empty disk, so fetch the repo (the starter CSV lives inside it)
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # Local run: walk up from the notebook folder to the repo root
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - are you at the repo root?"

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "| clients:", df["client_id"].nunique())

# One number per lane that tests its premise
down = df["trend_direction"].eq("down")
elig = df[(df["impressions_90d"] >= 500) & (df["avg_position"].between(0.1, 20))]
print(f"Lane 2 premise - share of pages with impressions down >20%: {down.mean():.1%}")
print(f"Lane 4 premise - eligible pages (>=500 impr, pos 1-20) with CTR < 0.5%: {(elig['ctr'] < 0.5).mean():.1%} of {len(elig):,}")
print(f"AI direction   - pages with any AI session: {(df['ai_sessions_90d'] > 0).mean():.1%} ({(df['ai_sessions_90d'] > 0).sum():,} pages)")


rows: 30000 | clients: 32
Lane 2 premise - share of pages with impressions down >20%: 54.2%
Lane 4 premise - eligible pages (>=500 impr, pos 1-20) with CTR < 0.5%: 81.2% of 12,023
AI direction   - pages with any AI session: 6.4% (1,930 pages)


## 2. The question: decision, action, cost of a wrong call

- **Decision improved:** which pages a reviewer opens first when they can only look at a limited number per cycle.
- **Who acts:** a content owner or editor at the client. **Action:** review the page and choose one of *refresh*, *rewrite title/meta*, *expand*, *merge*, or *monitor*.
- **Cost of a wrong recommendation:**
  - *False positive* (a page is ranked high but was not really in trouble): wasted reviewer time, and a risk of editing a page that was fine. This is the cost I will weigh most, because review capacity is the scarce resource.
  - *False negative* (a page that really was losing visibility never reaches the queue): lost traffic that nobody looked at. This is harder to see, so I will report recall or a missed-decline check next to precision.
- **Why data or ML can help at all, and why this is not just "train a model":** A plain rule (for example "flag every page that dropped") already exists and already flags most of the inventory, so it does not prioritise. What is left is a ranking problem: many signals (volume, position, age, engagement, size of the drop) have to be weighed together. The honest test is whether a learned ranking beats a transparent rule on a metric that matches the real workflow (precision@K for the reviewer's K), validated on clients the model has not seen. If it does not beat the rule, that is a result too, and the rule ships.

In [4]:
# Capacity arithmetic: what a reviewer can do vs. what a simple flag produces
K = 50  # assumed reviewer capacity per cycle (a planning assumption, not a measured fact)
flagged = df[down & (df["impressions_90d"] >= 100)]
print(f"Pages flagged by 'declining with demand' (down, >=100 impressions): {len(flagged):,}")
print(f"Reviewer capacity K = {K}  ->  the flag alone leaves {len(flagged)/K:,.0f}x more pages than can be reviewed")

Pages flagged by 'declining with demand' (down, >=100 impressions): 13,152
Reviewer capacity K = 50  ->  the flag alone leaves 263x more pages than can be reviewed


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter CSV (30,000 pages, 32 clients, trailing 90 days), run in the next cell:

1. **Declining is the majority, not the exception.** 54.2% of pages are labelled `down`. A flag that fires on more than half the inventory cannot decide what to review first.
2. **Demand is concentrated.** The top 10% of pages by impressions hold about 70% of all impressions, so a small part of the inventory carries most of what is at stake.
3. **Loss is concentrated too.** Among declining pages, the top 10% by impressions lost (last 30 days vs the 30 days before) account for about 69% of all impressions lost. Reviewing in the right order matters.

Bonus check: the guide's hand-written `stale_visible_page` rule (not updated in 180+ days and at least 500 impressions) fires on only 17 pages, so on this slice it is almost silent. Together these suggest a fixed rule is too loud in one place and too quiet in another, which is what a ranking can improve. This describes the starter slice only, not the full warehouse.

In [5]:
# 1) share of pages labelled 'down'
print(f"1) Pages with trend_direction == 'down': {down.mean():.1%} ({down.sum():,} of {len(df):,})")

# 2) concentration of demand
imp = df["impressions_90d"].sort_values(ascending=False)
top10 = imp.head(int(len(imp) * 0.10)).sum() / imp.sum()
print(f"2) Top 10% of pages hold {top10:.1%} of all 90-day impressions")

# 3) concentration of loss among declining pages
lost = (df["impressions_prev_30d"] - df["impressions_last_30d"])[down].sort_values(ascending=False)
top10_lost = lost.head(int(len(lost) * 0.10)).sum() / lost.sum()
print(f"3) Among declining pages, the top 10% account for {top10_lost:.1%} of impressions lost")

# bonus: the existing stale-visible rule
stale = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"Bonus) 'stale_visible_page' rule fires on {len(stale)} pages")

1) Pages with trend_direction == 'down': 54.2% (16,262 of 30,000)
2) Top 10% of pages hold 70.2% of all 90-day impressions
3) Among declining pages, the top 10% account for 68.7% of impressions lost
Bonus) 'stale_visible_page' rule fires on 17 pages


## 4. Careful words: what I can and can't claim

**What this work can say** (observed, directional, decision-support):
- "We observed that impressions fell by more than 20% on X% of pages between two 30-day windows."
- "This ranking puts pages with larger, better-supported losses earlier; on held-out clients it reached precision@K of Y versus Z for the rule." (Numbers to come from later weeks.)
- "These pages are good candidates for human review first."

**What this work will never say:**
- That a refresh *caused* or *will cause* a recovery. This data is observational; it has no experiment.
- That I know why Google moved a page, or that I found a ranking factor.
- That a page with no AI sessions is invisible to AI tools; the field only counts click-throughs.
- That the current `down` label is ground truth. It is a proxy built from one 30-day comparison, and the ranking should eventually be tested on a later window.

**Known weaknesses I must check later:** the `down` label can also come from seasonality, consolidation with a sibling page, or low-volume noise (guide section 7). The next cell shows how large two of those risks already look in the starter data: about a third of `down` pages started from under 100 impressions (noise-prone), and the share of pages that are `down` ranges from 0% to 94% across clients with 100+ pages, which hints at client-level effects such as seasonality or site changes.

In [6]:
# How strong are the noise / client-level risks behind the 'down' label?
low_base = df[down & (df["impressions_prev_30d"] < 100)]
print(f"'down' pages whose previous-30d impressions were under 100: {len(low_base) / down.sum():.1%}")

by_client = df.groupby("client_id").agg(n=("content_id", "size"), down=("trend_direction", lambda s: (s == "down").mean()))
by_client = by_client[by_client["n"] >= 100]
print(f"Clients with >=100 pages: {len(by_client)}")
print(f"Share of pages 'down' per client: min {by_client['down'].min():.0%}, median {by_client['down'].median():.0%}, max {by_client['down'].max():.0%}")

'down' pages whose previous-30d impressions were under 100: 31.8%
Clients with >=100 pages: 24
Share of pages 'down' per client: min 0%, median 55%, max 94%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.